# Naive Bayes

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Naive Bayes uses **Bayes' theorem** to compute the probability of each class after seeing the features:

$$P(c\mid x)=\frac{P(x\mid c)\,P(c)}{P(x)}$$

- $P(c)$: the **prior**, before seeing the data (e.g. 30% of emails are spam).
- $P(x\mid c)$: the **likelihood**, how common these features are in class $c$.
- $P(c\mid x)$: the **posterior**, after seeing the data.

## The "naive" assumption
Computing $P(x_1,\dots,x_d\mid c)$ is hard. We assume the features are **independent given the class**:

$$P(c\mid x)\propto P(c)\prod_{j=1}^{d}P(x_j\mid c)$$

This is usually false ("free" and "prize" appear together in spam), yet the model works very well in practice, because only the **order** of the probabilities matters for the decision.

## A full example: a spam filter
Of 100 emails, 30 are spam. A new email contains "free" and "meeting":

| Word | P(word ∣ spam) | P(word ∣ normal) |
|---|---|---|
| free | 0.60 | 0.05 |
| meeting | 0.10 | 0.40 |

- Spam: $0.3\times0.6\times0.1=0.018$
- Normal: $0.7\times0.05\times0.4=0.014$

Normalize: $P(\text{spam}\mid x)=0.018/(0.018+0.014)=0.56$ → **spam**.



In [ ]:
# The spam example, step by step
prior = {"spam": 0.3, "normal": 0.7}
like = {"spam": {"free": 0.60, "meeting": 0.10}, "normal": {"free": 0.05, "meeting": 0.40}}
email = ["free", "meeting"]

score = {}
for c in prior:
    s = prior[c]
    for w in email:
        s *= like[c][w]
    score[c] = s
total = sum(score.values())
for c in score:
    print(c, "score =", round(score[c], 4), " posterior =", round(score[c] / total, 3))



## Three common kinds
| Kind | Features | $P(x_j\mid c)$ | Use |
|---|---|---|---|
| `GaussianNB` | Continuous | Normal distribution with each class's mean and variance | Numeric data |
| `MultinomialNB` | Counts | Word frequency | Text classification |
| `BernoulliNB` | 0/1 | Word present or not | Short texts |

> [Gaussian naive Bayes in 1-D: each class density times its prior, and the posterior crossing 0.5.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/nb_gauss.png)

*Gaussian naive Bayes in 1-D: each class density times its prior, and the posterior crossing 0.5.*

## Laplace smoothing
If a word never appeared in spam during training, $P=0$ and the whole product becomes zero. With `alpha=1` we add 1 to every count:

$$P(w\mid c)=\frac{\text{count}(w,c)+\alpha}{\text{count}(c)+\alpha V}$$

[[lab:cls_nb]]



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

texts = ["win a free prize now", "free money click now", "meeting at ten tomorrow", "project meeting notes",
         "claim your free gift", "lunch with the team tomorrow", "cheap prize offer now", "notes from the team meeting"]
labels = ["spam", "spam", "normal", "normal", "spam", "normal", "spam", "normal"]

model = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0)).fit(texts, labels)
for t in ["free prize meeting", "team lunch tomorrow", "click now for money"]:
    p = model.predict_proba([t])[0]
    print(f"{t!r:25} -> {model.predict([t])[0]:7s}", dict(zip(model.classes_, p.round(3))))



| Pros | Cons |
|---|---|
| Extremely fast, even with millions of features | The independence assumption |
| Works with little data | Probabilities pushed near 0 or 1 (poorly calibrated) |
| A great baseline for text | Simple boundaries |

### Exercise

Evaluate GaussianNB on breast cancer with 5 folds and compare it with LogisticRegression.

In [ ]:
# Your answer here